# Delta Lake MERGE Implementation

**Objective:** Perform incremental data processing using Delta Lake on the Sample Superstore dataset, using PySpark on Databricks.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## Step 1 - Load the dataset into a Delta table

In [ ]:
csv_path = "/FileStore/tables/Sample_Superstore.csv"

df = spark.read.csv(csv_path, header=True, inferSchema=True, escape='"')
df.count()

In [ ]:
df.printSchema()

In [ ]:
display(df.limit(5))

In [ ]:
delta_path = "/FileStore/tables/delta/superstore"

df.write.format("delta").mode("overwrite").save(delta_path)
spark.read.format("delta").load(delta_path).count()

## Step 2 - Basic cleaning

In [ ]:
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [ ]:
print(df.count(), df.dropDuplicates().count())

In [ ]:
string_cols = [f.name for f in df.schema.fields if f.dataType.typeName() == "string"]

for c in string_cols:
    df = df.withColumn(c, F.trim(F.col(c)))

In [ ]:
df = df.withColumn("Order Date", F.to_date("Order Date", "M/d/yyyy"))
df = df.withColumn("Ship Date", F.to_date("Ship Date", "M/d/yyyy"))

df.select("Order Date", "Ship Date").show(3)

In [ ]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(
    delta_path
)
spark.read.format("delta").load(delta_path).count()

## Step 3 - Simulate an incremental batch


In [ ]:
update_ids = [
    r["Row ID"]
    for r in df.select("Row ID").orderBy(F.rand(seed=42)).limit(15).collect()
]

df_updates = df.filter(F.col("Row ID").isin(update_ids))
df_updates = (
    df_updates.withColumn("Quantity", F.col("Quantity") + 1)
    .withColumn("Discount", F.round(F.col("Discount") + 0.1, 2))
    .withColumn("Sales", F.round(F.col("Sales") * 1.05, 2))
    .withColumn("Profit", F.round(F.col("Profit") * 0.9, 2))
)

df_updates.select("Row ID", "Quantity", "Discount", "Sales", "Profit").show(5)

In [ ]:
max_id = df.agg(F.max("Row ID")).collect()[0][0]

w = Window.orderBy(F.monotonically_increasing_id())
new_rows = df.orderBy(F.rand(seed=7)).limit(10)
new_rows = new_rows.withColumn("rn", F.row_number().over(w))
new_rows = new_rows.withColumn("Row ID", F.col("rn") + F.lit(max_id))
new_rows = new_rows.withColumn(
    "Order ID",
    F.concat(F.lit("CA-2017-NEW"), F.lpad(F.col("rn").cast("string"), 3, "0")),
)
new_rows = new_rows.withColumn("Order Date", F.lit("2017-12-01").cast("date"))
new_rows = new_rows.withColumn("Ship Date", F.lit("2017-12-04").cast("date"))
new_rows = new_rows.drop("rn")

new_rows.select("Row ID", "Order ID", "Order Date", "Sales").show()

In [ ]:
df_incremental = df_updates.unionByName(new_rows)
df_incremental.count()

In [ ]:
dupe_rows = df_incremental.limit(2)
df_incremental = df_incremental.unionByName(dupe_rows)

print(
    "rows before dedup:",
    df_incremental.count(),
    "| distinct:",
    df_incremental.dropDuplicates().count(),
)

df_incremental = df_incremental.dropDuplicates()
df_incremental.count()

## Step 4 - MERGE into the Delta table

In [ ]:
delta_table = DeltaTable.forPath(spark, delta_path)
rows_before = delta_table.toDF().count()
rows_before

In [ ]:
(
    delta_table.alias("target")
    .merge(df_incremental.alias("source"), "target.`Row ID` = source.`Row ID`")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
delta_table.history(1).select("version", "operation", "operationMetrics").show(
    truncate=False
)

## Step 5 - Validation

Checking row counts, making sure Row ID is still unique after the merge, and spot-checking that the updated rows actually picked up the new values.

In [ ]:
final_df = delta_table.toDF()
rows_after = final_df.count()

print("rows before:", rows_before)
print("rows after:", rows_after)
print("net new rows:", rows_after - rows_before)

In [ ]:
final_df.groupBy("Row ID").count().filter("count > 1").count()

In [ ]:
check_id = update_ids[0]

print("updated row, from the incremental batch:")
df_incremental.filter(F.col("Row ID") == check_id).select(
    "Row ID", "Quantity", "Discount", "Sales", "Profit"
).show()

print("same row, read back from the delta table after merge:")
final_df.filter(F.col("Row ID") == check_id).select(
    "Row ID", "Quantity", "Discount", "Sales", "Profit"
).show()

In [ ]:
final_df.filter(F.col("Row ID") == max_id + 1).select(
    "Row ID", "Order ID", "Order Date", "Sales"
).show()

Counts line up, Row ID has no duplicates, the updated rows show the bumped numbers, and the new order rows are sitting in the table. MERGE worked as expected.

## Step 6 - Final dataset and summary

In [ ]:
final_df.count()

In [ ]:
(
    final_df.groupBy("Category")
    .agg(
        F.round(F.sum("Sales"), 2).alias("Sales"),
        F.round(F.sum("Profit"), 2).alias("Profit"),
    )
    .orderBy("Category")
    .show()
)

In [ ]:
display(final_df.groupBy("Category").agg(F.sum("Sales").alias("Sales")))

In [ ]:
display(final_df.orderBy(F.desc("Row ID")).limit(5))